
✨ 02 — Routes + Templates: Jinja's Magic!

**Topic 13.03 | Other_Libraries — Part 3 of 8 (templates notebook)**

🌍 **Hook:** Lentils & potatoes served with the SAME plating on a "master-platter": the
ingredients keep changing, the FRAME stays the same! On the web, we call that a
template: the page-frame (header/nav/footer) once, the data fresh every day.
Jinja = Flask's plating wizard 🎨

📖 **Jinja-symbols cheat:**
- `{{ name }}` = stick in a variable (the value lands on the plate!)
- `{% for x in items %}` … `{% endfor %}` = loop-platter
- `{% if cond %}` … `{% endif %}` = conditional garnishing
- `{% extends "base.html" %}` + `{% block %}` = build a page from the master-frame (inheritance!)


## 🐣 Level 0 — `render_template_string`: In-Notebook Templates! 🥟

In [1]:
from flask import Flask, render_template_string, request

app = Flask("catalog-app 🛍️")
PRODUCTS = [
    {"id": 1, "name": "Pen",      "price": 20,  "stock": 50},
    {"id": 2, "name": "Notebook", "price": 50,  "stock": 30},
    {"id": 3, "name": "Bag",      "price": 600, "stock": 5},
    {"id": 4, "name": "Bottle",   "price": 90,  "stock": 12},
]

@app.route("/")
def home():
    tpl = """
    <h1>🛍️ BazarKaap</h1>
    <p>dukaan me {{ items|length }} items!</p>
    <ul>{% for p in items %}<li>{{ p.name }} — ₹{{ p.price }}</li>{% endfor %}</ul>
    """
    return render_template_string(tpl, items=PRODUCTS)

c = app.test_client()
html = c.get("/").data.decode()
print(html.strip())
# Expected OUTPUT:
# <h1>🛍️ BazarKaap</h1>
#     <p>dukaan me 4 items!</p>
#     <ul><li>Pen — ₹20</li><li>Notebook — ₹50</li><li>Bag — ₹600</li><li>Bottle — ₹90</li></ul>

<h1>🛍️ BazarKaap</h1>
    <p>dukaan me 4 items!</p>
    <ul><li>Pen — ₹20</li><li>Notebook — ₹50</li><li>Bag — ₹600</li><li>Bottle — ₹90</li></ul>


## 🐥 Level 1 — If/Else Garnishing + Filters 🌶️

In [2]:
app_alert = Flask("dhaba-alerts 🛎️")   # restart-roop (pichhla app freeze!)

@app_alert.route("/stock_alerts")
def stock_alerts():
    tpl = """
    {% for p in items -%}
      {% if p.stock < 10 -%}
        <li>⚠️ {{ p.name | upper }} — sirf {{ p.stock }} bache!</li>
      {%- else -%}
        <li>✅ {{ p.name }} — {{ p.stock }} ok</li>
      {%- endif -%}
    {%- endfor %}
    """
    return "<ul>" + render_template_string(tpl, items=PRODUCTS) + "</ul>"

r = app_alert.test_client().get("/stock_alerts")
print(r.data.decode())
# Expected OUTPUT:
# <ul><li>✅ Pen — 50 ok</li>
#         <li>✅ Notebook — 30 ok</li>
#         <li>⚠️ BAG — sirf 5 bache!</li>
#         <li>✅ Bottle — 12 ok</li>
# </ul>

<ul>
    <li>✅ Pen — 50 ok</li><li>✅ Notebook — 30 ok</li><li>⚠️ BAG — sirf 5 bache!</li><li>✅ Bottle — 12 ok</li>
    </ul>



## 🐔 Level 2 — REAL Template Inheritance (with DictLoader!) 🏛️

In real projects you keep files like this: `templates/base.html`, `templates/page.html`.
How do you learn in a notebook without files? **DictLoader** = a fake of a whole
"templates folder" in memory! (all-real, only the disk-read is a dummy!) 🎭


In [3]:
from jinja2 import DictLoader
from flask import render_template

app2 = Flask("shop-frame 🏛️")
app2.jinja_loader = DictLoader({
    "base.html": """
        <html><body>
        <header>🛍️ BazarKaap — {{ tagline }}</header>
        {% block content %}{% endblock %}
        <footer>© 2026 BazarKaap | items: {{ items|length }}</footer>
        </body></html>""",
    "catalog.html": """
        {% extends "base.html" %}
        {% block content %}
        <table><tr><th>Item</th><th>₹</th></tr>
        {% for p in items %}<tr><td>{{ p.name }}</td><td>{{ p.price }}</td></tr>{% endfor %}
        </table>
        {% endblock %}""",
})

@app2.route("/catalog")
def catalog():
    return render_template("catalog.html", items=PRODUCTS, tagline="sab sasta, sab mast!")

c2 = app2.test_client()
html2 = c2.get("/catalog").data.decode()
print(html2)
# Expected OUTPUT (extends = base ka frame + catalog ka block dikhta!):
# <html><body>
#         <header>🛍️ BazarKaap — sab sasta, sab mast!</header>
#
#         <table><tr><th>Item</th><th>₹</th></tr>
#         <tr><td>Pen</td><td>20</td></tr><tr><td>Notebook</td><td>50</td></tr><tr><td>Bag</td><td>600</td></tr><tr><td>Bottle</td><td>90</td></tr>
#         </table>
#
#         <footer>© 2026 BazarKaap | items: 4</footer>
#         </body></html>


        
        <html><body>
        <header>🛍️ BazarKaap — sab sasta, sab mast!</header>
        
        <table><tr><th>Item</th><th>₹</th></tr>
        <tr><td>Pen</td><td>20</td></tr><tr><td>Notebook</td><td>50</td></tr><tr><td>Bag</td><td>600</td></tr><tr><td>Bottle</td><td>90</td></tr>
        </table>
        
        <footer>© 2026 BazarKaap | items: 4</footer>
        </body></html>



---
## ⚠️ 5 Classic Mistakes (Templates)
| # | ❌ Mistake | ✅ Correct |
|---|---|---|
| 1 | `"{{ p.name}}"` with the space in the wrong place | write it consistently |
| 2 | `{% extends %}` not on the first line | extends is the VERY-FIRST part! |
| 3 | block-name mismatch (many a shut door!) | `{% block content %}` same on both sides |
| 4 | int/String compare in jinja (`{{ p.stock < "10" }}`) | types done straight (int-int!) |
| 5 | hardcoding the form-action | `url_for('add')` works in templates too! |

## 📌 Recap — 30 Second
- `{{ var }}` · `{% for/if %}` · `|length`, `|upper` filters = the Jinja environment
- `render_template_string` in the notebook; use `render_template` when you have files
- DictLoader + `extends/block` = inheritance from HOME — master-level! 🏛️

**Next:** `task.ipynb` — 7 shop-tickets for BazarKaap + the SHOPKEEPER BOSS! 🛍️🎫
